In [1]:
import os
import re
import json
import shutil
import random
from collections import defaultdict

import numpy as np
from PIL import Image
from tqdm import tqdm

# ----------------------------- CONFIG -----------------------------
SRC_DIR  = "/kaggle/input/datasets/kangaonkaggle/monuseg-preprocessed/MoNuSeg"   # contains training/ and testing/
DEST_DIR = "./monuseg_preprocessed"       # must be DIFFERENT from SRC_DIR

VAL_FRAC = 0.20                    # fraction of TRAINING IMAGES going to validation
SEED     = 42
GROUP_BY = "patient"               # "patient" (recommended) | "image"

TRANSFER_MODE = "copy"             # "copy" | "move" | "symlink"
OVERWRITE     = False              # refuse to write into a non-empty DEST_DIR unless True

IMG_EXT = ".png"
LAB_EXT = ".npy"
# ------------------------------------------------------------------

PATIENT_RE = re.compile(r"(TCGA-[A-Za-z0-9]{2}-[A-Za-z0-9]{4})")


def transfer(src, dst):
    if TRANSFER_MODE == "copy":
        shutil.copy2(src, dst)
    elif TRANSFER_MODE == "move":
        shutil.move(src, dst)
    elif TRANSFER_MODE == "symlink":
        if os.path.lexists(dst):
            os.remove(dst)
        os.symlink(os.path.abspath(src), dst)
    else:
        raise ValueError(f"Unknown TRANSFER_MODE: {TRANSFER_MODE}")


def list_stems(folder, ext):
    return sorted(os.path.splitext(f)[0] for f in os.listdir(folder) if f.endswith(ext))


def get_pairs(part_dir):
    """Return sorted stems; require that every image has a label and vice versa."""
    img_dir, lab_dir = os.path.join(part_dir, "images"), os.path.join(part_dir, "labels")
    for d in (img_dir, lab_dir):
        if not os.path.isdir(d):
            raise FileNotFoundError(f"Missing directory: {d}")
    imgs, labs = list_stems(img_dir, IMG_EXT), list_stems(lab_dir, LAB_EXT)
    if set(imgs) != set(labs):
        raise ValueError(
            f"Image/label mismatch in {part_dir}\n"
            f"  images without labels: {sorted(set(imgs) - set(labs))}\n"
            f"  labels without images: {sorted(set(labs) - set(imgs))}"
        )
    if not imgs:
        raise ValueError(f"No {IMG_EXT} images found in {img_dir}")
    return imgs


def group_key(stem):
    if GROUP_BY == "image":
        return stem
    m = PATIENT_RE.search(stem)
    return m.group(1) if m else None


def build_groups(stems):
    groups, unmatched = defaultdict(list), []
    for s in stems:
        k = group_key(s)
        if k is None:
            unmatched.append(s)
            k = s                      # fallback: treat as its own group
        groups[k].append(s)
    return dict(groups), unmatched


def split_groups(groups, val_frac, seed):
    """Shuffle groups, then add to val only while it moves val closer to the target image count."""
    rng = random.Random(seed)
    keys = sorted(groups)
    rng.shuffle(keys)
    total = sum(len(groups[k]) for k in keys)
    target = round(total * val_frac)
    val_keys, n = [], 0
    for k in keys:
        size = len(groups[k])
        if abs(target - (n + size)) < abs(target - n):
            val_keys.append(k)
            n += size
    val_set = set(val_keys)
    train_keys = [k for k in keys if k not in val_set]
    assert val_keys and train_keys, "Empty train or val split; check VAL_FRAC / data."
    return sorted(train_keys), sorted(val_keys)


def describe(name, stems, part_dir):
    """Image count + total nuclei instances (non-zero IDs) as a balance sanity check."""
    n_nuclei = 0
    for s in stems:
        m = np.load(os.path.join(part_dir, "labels", s + LAB_EXT))
        assert m.ndim == 2, f"{s}: expected 2D label map, got {m.shape}"
        n_nuclei += int((np.unique(m) > 0).sum())
    print(f"{name:<6} {len(stems):>3} images | {n_nuclei:>6} nuclei | "
          f"{n_nuclei / max(len(stems), 1):.0f} nuclei/image")


def main():
    src_train = os.path.join(SRC_DIR, "training")
    src_test  = os.path.join(SRC_DIR, "testing")

    # ---- safety checks (no rmtree on purpose) ----
    if os.path.abspath(SRC_DIR) == os.path.abspath(DEST_DIR):
        raise ValueError("DEST_DIR must differ from SRC_DIR.")
    if os.path.isdir(DEST_DIR) and os.listdir(DEST_DIR) and not OVERWRITE:
        raise FileExistsError(f"{DEST_DIR} exists and is not empty. Set OVERWRITE=True or pick another path.")

    train_stems = get_pairs(src_train)
    test_stems  = get_pairs(src_test)
    assert not (set(train_stems) & set(test_stems)), "Same filename in train and test!"

    # ---- show what the data actually looks like (the README claims grayscale; verify) ----
    probe = Image.open(os.path.join(src_train, "images", train_stems[0] + IMG_EXT))
    probe_lab = np.load(os.path.join(src_train, "labels", train_stems[0] + LAB_EXT))
    print(f"Image probe: mode={probe.mode}, size={probe.size}")
    print(f"Label probe: shape={probe_lab.shape}, dtype={probe_lab.dtype}, max id={probe_lab.max()}")

    # ---- group + split ----
    groups, unmatched = build_groups(train_stems)
    if GROUP_BY == "patient" and unmatched:
        print(f"[Warning] {len(unmatched)}/{len(train_stems)} files have no TCGA patient ID in the "
              f"filename; each is treated as its own group, so patient leakage is NOT prevented for them.")
        print(f"          e.g. {unmatched[:3]}")
    print(f"{len(train_stems)} training images -> {len(groups)} groups (GROUP_BY='{GROUP_BY}')")

    train_keys, val_keys = split_groups(groups, VAL_FRAC, SEED)
    tr_stems = sorted(s for k in train_keys for s in groups[k])
    va_stems = sorted(s for k in val_keys for s in groups[k])
    assert not (set(tr_stems) & set(va_stems)), "Train/val overlap!"
    assert len(tr_stems) + len(va_stems) == len(train_stems), "Images lost or duplicated in split!"

    # ---- warn if test patients also appear in train/val (dataset property, not a script bug) ----
    if GROUP_BY == "patient":
        test_patients = {group_key(s) for s in test_stems} - {None}
        trainval_patients = set(train_keys) | set(val_keys)
        shared = test_patients & trainval_patients
        if shared:
            print(f"[Warning] Patients present in both test and train/val: {sorted(shared)}")

    print("-" * 50)
    describe("Train", tr_stems, src_train)
    describe("Val", va_stems, src_train)
    describe("Test", test_stems, src_test)
    print(f"Val groups: {val_keys}")
    print("-" * 50)

    # ---- write ----
    jobs = [("training", src_train, tr_stems),
            ("validation", src_train, va_stems),
            ("testing", src_test, test_stems)]
    for part, src_part, stems in jobs:
        for sub in ("images", "labels"):
            os.makedirs(os.path.join(DEST_DIR, part, sub), exist_ok=True)
        for s in tqdm(stems, desc=part):
            transfer(os.path.join(src_part, "images", s + IMG_EXT), os.path.join(DEST_DIR, part, "images", s + IMG_EXT))
            transfer(os.path.join(src_part, "labels", s + LAB_EXT), os.path.join(DEST_DIR, part, "labels", s + LAB_EXT))

    with open(os.path.join(DEST_DIR, "split.json"), "w") as f:
        json.dump({"seed": SEED, "group_by": GROUP_BY, "val_frac": VAL_FRAC,
                   "train": tr_stems, "val": va_stems, "test": test_stems,
                   "val_groups": val_keys}, f, indent=2)

    # ---- verify what was actually written ----
    for part, _, stems in jobs:
        for sub, ext in (("images", IMG_EXT), ("labels", LAB_EXT)):
            n = len([x for x in os.listdir(os.path.join(DEST_DIR, part, sub)) if x.endswith(ext)])
            assert n == len(stems), f"{part}/{sub}: expected {len(stems)} files, found {n}"
    print("DONE. Output at:", DEST_DIR)


if __name__ == "__main__":
    main()

Image probe: mode=RGB, size=(1000, 1000)
Label probe: shape=(1000, 1000), dtype=int32, max id=480
37 training images -> 37 groups (GROUP_BY='patient')
--------------------------------------------------
Train   30 images |  20305 nuclei | 677 nuclei/image
Val      7 images |   3825 nuclei | 546 nuclei/image
Test    14 images |   6697 nuclei | 478 nuclei/image
Val groups: ['TCGA-50-5931', 'TCGA-AR-A1AS', 'TCGA-B0-5710', 'TCGA-FG-A87N', 'TCGA-G9-6336', 'TCGA-G9-6363', 'TCGA-HE-7128']
--------------------------------------------------


testing: 100%|██████████| 14/14 [00:00<00:00, 27.07it/s]

DONE. Output at: ./monuseg_preprocessed


In [2]:
main()

FileExistsError: ./monuseg_preprocessed exists and is not empty. Set OVERWRITE=True or pick another path.